In [1]:
import pandas as pd
import numpy as np

In [2]:
apis=pd.read_excel("apis_dataset_dirty_real_world.xlsx")

In [5]:
apis.head()
apis.tail()
apis.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   api_id        5000 non-null   int64
 1   api_name      5000 non-null   str  
 2   api_category  4607 non-null   str  
 3   endpoint      5000 non-null   str  
 4   version       4602 non-null   str  
 5   owner_team    4600 non-null   str  
 6   status        5000 non-null   str  
dtypes: int64(1), str(6)
memory usage: 273.6 KB


In [6]:
apis.isnull().sum()

api_id            0
api_name          0
api_category    393
endpoint          0
version         398
owner_team      400
status            0
dtype: int64

In [7]:
apis['api_category']=apis['api_category'].fillna("Unknown")
apis['version']=apis['version'].fillna("Unknown")
apis['owner_team']=apis['owner_team'].fillna("Unknown")

In [8]:
apis.isnull().sum()

api_id          0
api_name        0
api_category    0
endpoint        0
version         0
owner_team      0
status          0
dtype: int64

In [11]:
apis.duplicated().sum()
apis[apis.duplicated(subset=['api_id'])]
apis[apis.duplicated(subset=['api_name'])]

,api_id,api_name,api_category,endpoint,version,owner_team,status
9,10,Snowflake Ingestion API,Analytics,https://snowflake.internal/api/v1/ingest/660,v1.4,Unknown,Active
11,12,Stripe Payments API,Payment,https://api.stripe.com/v3/charges/435,v1.0.0,Logistics,Active
16,17,Twilio SMS Gateway API,Notifications,https://api.twilio.com/2010-04-01/Accounts,v2.1.0,Unknown,Active
20,21,User Profile Core API,User,https://internal.platform/services/user-profile,v1.4,Logistics,Active
28,29,Okta Verify Service,Auth,https://okta.internal/api/v1/authn/913,v2-beta,Comms-Team,Active
...,...,...,...,...,...,...,...
4994,4995,User Profile Core API,User,https://internal.platform/services/user-pro...,v2-beta,Identity-IAM,Active
4995,4996,Twilio SMS Gateway Service,Notifications,https://api.twilio.com/2010-04-01/Accounts,2024-01-01,FinTech-Core,Active
4996,4997,Analytics Engine #53,Analytics,https://internal.api/v1/analytics/action,2024-01-01,SecOps-Compliance,Active
4997,4998,Marketing Engine #85,Marketing,https://internal.api/v1/marketing/action,v1-alpha,SecOps-Compliance,Active


In [13]:
apis['api_name'].nunique()
apis['api_category'].unique()

<StringArray>
[           'IAM',           'user',      'Analytics',           'Maps',
        'Payment',        'Unknown',     'Compliance',  'Notifications',
       'Shipping',            'CRM',           'User',      'Marketing',
           'Auth',       'Security',  'notifications',        'FinTech',
           'auth',       'PAYMENTS',          'Comms', 'Authentication',
      'CORE_USER',      'SMS-Email',        'Profile',        'payment']
Length: 24, dtype: str

In [14]:
apis['api_category']=apis['api_category'].replace({
  'auth':'Auth',
  'Authentication':'Auth',
  
  'PAYMENTS': 'Payment',
    'payment': 'Payment',

    'notifications': 'Notifications',

    'user': 'User',
    'CORE_USER': 'User',
    'Profile': 'User',

    'SMS-Email': 'Comms'
  
})

In [15]:
apis['api_category'].unique()

<StringArray>
[          'IAM',          'User',     'Analytics',          'Maps',
       'Payment',       'Unknown',    'Compliance', 'Notifications',
      'Shipping',           'CRM',     'Marketing',          'Auth',
      'Security',       'FinTech',         'Comms']
Length: 15, dtype: str

In [32]:
#endpoint
apis['endpoint'].isnull().sum()
apis['endpoint']=apis['endpoint'].str.lower()
apis['endpoint']=apis['endpoint'].str.strip()
apis[~apis['endpoint'].str.startswith('/')]
apis['service_name']=apis['endpoint'].str.split('/').str[2].str.split('.').str[0]
apis['service_name']

0            okta
1        internal
2       snowflake
3            maps
4           api-m
          ...    
4995          api
4996     internal
4997     internal
4998          api
4999     internal
Name: service_name, Length: 5000, dtype: object

In [40]:
apis['endpoint'] = apis['endpoint'].str.split('?').str[0]
apis['endpoint'].str.startswith('https').value_counts()
apis['endpoint']


0                  https://okta.internal/api/v1/authn/467
1       https://internal.platform/services/user-profil...
2                https://snowflake.internal/api/v1/ingest
3       https://maps.googleapis.com/maps/api/geocode/json
4         https://api-m.paypal.com/v2/checkout/orders/106
                              ...                        
4995           https://api.twilio.com/2010-04-01/accounts
4996             https://internal.api/v1/analytics/action
4997             https://internal.api/v1/marketing/action
4998                https://api.stripe.com/v3/charges/505
4999         https://internal.api/v1/notifications/action
Name: endpoint, Length: 5000, dtype: object

In [42]:
apis['version'].isnull().sum()
apis['version'].unique()

<StringArray>
[          'v1.4', '0.0.1-SNAPSHOT',   'test-version',        'Unknown',
             'v3',         'v1.0.0',         'v2.1.0',        'v2-beta',
     '2024-01-01',       'v1-alpha',       '2026_RC4']
Length: 11, dtype: str

In [54]:
apis['owner_team'].isnull().sum()
apis['owner_team'].unique()

apis['owner_team']=(apis['owner_team'].str.strip().str.lower())
apis['owner_team'] = apis['owner_team'].replace({
    'fintech-core': 'FinTech-Core',
    'identity-iam': 'Identity-IAM',
    'comms-team': 'Comms-Team',
    'growth-marketing': 'Growth-Marketing',
    'core-platform': 'Core-Platform',
    'secops-compliance': 'SecOps-Compliance',
    'logistics': 'Logistics',
    'data-eng': 'Data-Eng',
    'unknown': 'Unknown'
})


In [75]:
apis['status'].isnull().sum()
apis['status'].unique()
apis['status'] = (apis['status'].str.strip().str.lower())
apis['status'] = apis['status'].replace({
    'active': 'Active',
    'deprecated': 'Deprecated',
    'testing': 'Testing',
    'inactive': 'Inactive'
})
apis['status'].unique()

<StringArray>
['Deprecated', 'Active', 'Inactive', 'legacy-active']
Length: 4, dtype: str

In [77]:
apis.info()
apis.isnull().sum()
print(apis['api_category'].unique())
print(apis['owner_team'].unique())
print(apis['status'].unique())
print(apis['version'].unique())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   api_id        5000 non-null   int64 
 1   api_name      5000 non-null   str   
 2   api_category  5000 non-null   str   
 3   endpoint      5000 non-null   object
 4   version       5000 non-null   str   
 5   owner_team    5000 non-null   str   
 6   status        5000 non-null   str   
 7   service_name  5000 non-null   object
dtypes: int64(1), object(2), str(5)
memory usage: 312.6+ KB
<StringArray>
[          'IAM',          'User',     'Analytics',          'Maps',
       'Payment',       'Unknown',    'Compliance', 'Notifications',
      'Shipping',           'CRM',     'Marketing',          'Auth',
      'Security',       'FinTech',         'Comms']
Length: 15, dtype: str
<StringArray>
[     'FinTech-Core',      'Identity-IAM',        'Comms-Team',
  'Growth-Marketing',     'Core-Platform', 'SecOps-

In [78]:
apis.to_csv('cleaned_api_dataset.csv',index=False)